In [42]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
import os
os.chdir(r'C:\Kaggle_Competition\Playground\S6E5-F1-Pit-Stops')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import gc
from sklearn.manifold import TSNE
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import optuna
# Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler
import plotly.express as px

# Externals
from src.utils import *
import config
import warnings
warnings.filterwarnings('ignore')

In [44]:
train = read_csv_file('data/raw/train.csv')
test = read_csv_file('data/raw/test.csv')
orig = read_csv_file('data/orig/f1_strategy_dataset_v4.csv')
orig = orig.dropna(axis=0)

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)
orig = reduce_mem_usage(orig)

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

print(f'Shape of train data: {train.shape}')
print(f'Shape of test data: {test.shape}')
print(f'Shape of orig data: {orig.shape}')

Shape of train data: (439140, 16)
Shape of test data: (188165, 15)
Shape of orig data: (101305, 16)


In [45]:
# Data
train_features = train.drop(['id', config.TARGET], axis=1)
target = train[config.TARGET].values
X_test = test.drop('id', axis=1)

orig_features = orig.drop([config.TARGET], axis=1)
orig_features = orig_features.dropna(axis=0)

### Blending

In [46]:
# HistGBM
oof_proba_hist = read_csv_file(r'artifacts\oof\HISTGBM_v2_oof_proba.csv')
oof_proba_hist = oof_proba_hist.iloc[:, 1:]

test_proba_hist = read_csv_file(r'artifacts\test_proba\HISTGBM_v2_test_proba.csv')
test_proba_hist = test_proba_hist.iloc[:, 1:]

# XGBM
oof_proba_xgbm = read_csv_file(r'artifacts\oof\XGB_v6_oof_proba.csv')
oof_proba_xgbm = oof_proba_xgbm.iloc[:, 1:]

test_proba_xgbm = read_csv_file(r'artifacts\test_proba\XGB_v6_test_proba.csv')
test_proba_xgbm = test_proba_xgbm.iloc[:, 1:]

# LGBM
oof_proba_lgbm = read_csv_file(r'artifacts\oof\LGBM_v4_seed42_oof_proba.csv')
oof_proba_lgbm = oof_proba_lgbm.iloc[:, 1:]

test_proba_lgbm = read_csv_file(r'artifacts\test_proba\LGBM_v4_seed42_test_proba.csv')
test_proba_lgbm = test_proba_lgbm.iloc[:, 1:]

In [47]:
# OOF
oof_hist = oof_proba_hist.values
oof_xgbm = oof_proba_xgbm.values
oof_lgbm = oof_proba_lgbm.values

# TEST
test_hist = test_proba_hist.values
test_xgbm = test_proba_xgbm.values
test_lgbm = test_proba_lgbm.values

In [48]:
score_hist = roc_auc_score(target, oof_hist)
score_xgbm = roc_auc_score(target, oof_xgbm)
score_lgbm = roc_auc_score(target, oof_lgbm)

print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")

HISTGBM alone: 0.945228
XGBM alone: 0.948816
LGBM alone: 0.948235


In [49]:
def objective(trial):
    w_hist = trial.suggest_float("w_hist", 0.0, 1.0)
    w_xgbm = trial.suggest_float("w_xgbm", 0.0, 1.0)
    w_lgbm = trial.suggest_float("w_lgbm", 0.0, 1.0)
    total = (
        w_hist + w_xgbm + w_lgbm
    )

    w_hist /= total
    w_xgbm /= total
    w_lgbm /= total

    preds = (
        w_hist * oof_hist +
        w_xgbm * oof_xgbm +
        w_lgbm * oof_lgbm
    )

    return roc_auc_score(target, preds)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective, n_trials=5000, show_progress_bar=True)

# -------- BEST WEIGHTS --------
best = study.best_params

total = (
    best["w_hist"] +
    best["w_xgbm"] +
    best["w_lgbm"]
)

w_hist = best["w_hist"] / total
w_xgbm = best["w_xgbm"] / total
w_lgbm = best["w_lgbm"] / total

# -------- PRINT --------
best_single = max(
    score_hist, score_xgbm, score_lgbm
)

print(f"\n{'='*40}")
print(f"HISTGBM alone: {score_hist:.6f}")
print(f"XGBM alone: {score_xgbm:.6f}")
print(f"LGBM alone: {score_lgbm:.6f}")
print(f"Best blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")
print(f"{'='*40}")

print(f"HIST weight: {w_hist:.4f} ({w_hist:.1%})")
print(f"XGBM weight: {w_xgbm:.4f} ({w_xgbm:.1%})")
print(f"LGBM weight: {w_lgbm:.4f} ({w_lgbm:.1%})")

[I 2026-05-12 23:31:01,563] A new study created in memory with name: no-name-efa2a47f-03e6-4a25-af50-3b16c98743e3


  0%|          | 0/5000 [00:00<?, ?it/s]

[I 2026-05-12 23:31:01,725] Trial 0 finished with value: 0.9491850509902781 and parameters: {'w_hist': 0.3745401188473625, 'w_xgbm': 0.9507143064099162, 'w_lgbm': 0.7319939418114051}. Best is trial 0 with value: 0.9491850509902781.
[I 2026-05-12 23:31:01,839] Trial 1 finished with value: 0.9475805129848685 and parameters: {'w_hist': 0.5986584841970366, 'w_xgbm': 0.15601864044243652, 'w_lgbm': 0.15599452033620265}. Best is trial 0 with value: 0.9491850509902781.
[I 2026-05-12 23:31:01,950] Trial 2 finished with value: 0.9494175295222433 and parameters: {'w_hist': 0.05808361216819946, 'w_xgbm': 0.8661761457749352, 'w_lgbm': 0.6011150117432088}. Best is trial 2 with value: 0.9494175295222433.
[I 2026-05-12 23:31:02,058] Trial 3 finished with value: 0.9480942111884029 and parameters: {'w_hist': 0.7080725777960455, 'w_xgbm': 0.020584494295802447, 'w_lgbm': 0.9699098521619943}. Best is trial 2 with value: 0.9494175295222433.
[I 2026-05-12 23:31:02,177] Trial 4 finished with value: 0.94747639

KeyboardInterrupt: 

In [ ]:
blend_oof = (
    w_hist * oof_hist +
    w_xgbm * oof_xgbm +
    w_lgbm * oof_lgbm
)

blend_test = (
    w_hist * test_hist +
    w_xgbm * test_xgbm +
    w_lgbm * test_lgbm
)

# -------- SAVE BLENDED FILES --------
blend_oof_df = pd.DataFrame(blend_oof, columns=['PitNextLap'])
blend_oof_df.insert(0, "id", train_ids)

save_csv_file(
    blend_oof_df,
    os.path.join(
        config.OOF_DIR,
        'blend_.csv'
    )
)

blend_test_df = pd.DataFrame(blend_test, columns=['PitNextLap'])
blend_test_df.insert(0, "id", test_ids)

save_csv_file(
    blend_test_df,
    os.path.join(
        config.TEST_PROBA_DIR,
        'blend_.csv'
    )
)